# Fixing the Blended Adaptive Threshold for Window=60

**The bug, precisely**: `adaptive_threshold_blended.ipynb`'s formula uses a
FIXED `K_ADAPTIVE=3.0` multiplier: `threshold = blended_mean + 3 * blended_std`.
At window=30, `mu_train + 3*sigma_train` happens to sit close enough to the
TRUE leak-free 99th-percentile threshold (`val_p99`) that this worked. At
window=60, it doesn't: calibration could only reach ~1.9-2.3% FPR against a
1% target, and applying the mechanism made both `cc1_test` and `drift_cc2`
WORSE than plain VAE-alone scoring.

**The fix**: stop assuming `k=3.0` is universal. Instead, DERIVE the k that
makes the formula's own global (no-local-history) endpoint reproduce the
actual empirical `val_p99` threshold exactly:

```
k_calibrated = (val_p99 - mu_train) / sigma_train
```

This is still fully leak-free (uses only `cc1_val`, same as every other
threshold in this project) — it just replaces an assumed constant with one
derived from the model's own error distribution, which is the correct fix
for a formula whose parametric assumption (roughly-Gaussian errors) stops
holding as well at larger window sizes.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle, joblib, os, copy
from collections import deque
from scipy import stats
from sklearn.metrics import precision_score, recall_score, f1_score

BASE      = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
DATA_DIR  = os.path.join(BASE, 'data', 'processed')
OUT_DIR   = os.path.join(BASE, 'experiments')
WINDOW_SIZE = 60
BUFFER_SIZE = 500

FEATURE_COLS = [
    'container_cpu_usage_seconds_rate', 'container_cpu_system_seconds_rate', 'container_cpu_user_seconds_rate',
    'container_memory_usage_bytes', 'container_memory_working_set_bytes', 'container_memory_rss', 'container_memory_cache',
]
SPLIT_FILES = {
    'cc1_train': 'cc1_train.csv', 'cc1_val': 'cc1_val.csv', 'cc1_test': 'cc1_test.csv',
    'drift_cc2': 'drift_complex_case2.csv',
}
ALL_SETS = ['cc1_test', 'drift_cc2']

class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Linear(hidden1, hidden2), nn.ReLU())
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden2), nn.ReLU(), nn.Linear(hidden2, hidden1), nn.ReLU(), nn.Linear(hidden1, input_dim))
    def encode(self, x):
        h = self.encoder(x); return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)
    def decode(self, z): return self.decoder(z)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar); return mu + torch.randn_like(std) * std
    def forward(self, x):
        mu, logvar = self.encode(x); z = self.reparameterize(mu, logvar); return self.decode(z), mu, logvar
    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval(); mu, _ = self.encode(x); return ((self.decode(mu) - x) ** 2).mean(dim=1)

w60_saved = pickle.load(open(os.path.join(OUT_DIR, 'window_size_60_results.pkl'), 'rb'))
meta60 = w60_saved['model_meta']
CLIP = meta60['clip']

base_model = VAE(meta60['input_dim'], meta60['hidden1'], meta60['hidden2'], meta60['latent_dim'])
base_model.load_state_dict(torch.load(os.path.join(OUT_DIR, 'vae_cc1_window60.pt'), map_location='cpu'))
base_model.eval()
pca = joblib.load(os.path.join(OUT_DIR, 'cc1_pca_window60.pkl'))['pca']

GLOBAL_MEAN, GLOBAL_STD, VAL_P99 = meta60['mu_train'], meta60['sigma_train'], meta60['val_p99']
K_OLD = 3.0
K_CALIBRATED = (VAL_P99 - GLOBAL_MEAN) / GLOBAL_STD
print(f'mu_train={GLOBAL_MEAN:.5f}  sigma_train={GLOBAL_STD:.5f}  val_p99={VAL_P99:.5f}')
print(f'Old fixed k = {K_OLD}  ->  global endpoint = {GLOBAL_MEAN + K_OLD*GLOBAL_STD:.5f}  (vs true val_p99 = {VAL_P99:.5f})')
print(f'Calibrated k = {K_CALIBRATED:.4f}  ->  global endpoint = {GLOBAL_MEAN + K_CALIBRATED*GLOBAL_STD:.5f}  (matches val_p99 exactly by construction)')

mu_train=0.28407  sigma_train=0.24760  val_p99=1.44733
Old fixed k = 3.0  ->  global endpoint = 1.02686  (vs true val_p99 = 1.44733)
Calibrated k = 4.6982  ->  global endpoint = 1.44733  (matches val_p99 exactly by construction)


## Step 1 — Rebuild window=60 data with per-window `cmdb_id`/timestamp

In [2]:
def build_windows_full(df, feature_cols, window_size, stride=1):
    data_arr_all, cmdb_all, ts_all, y_all, ft_all = [], [], [], [], []
    for cmdb_id, g in df.sort_values('timestamp').groupby('cmdb_id'):
        data_arr = g[feature_cols].values.astype(np.float32)
        is_gap = g['is_gap'].values
        labels = g['label'].values
        ftypes = g['failure_type'].values.astype(object)
        ts = g['timestamp'].values
        n = len(g)
        for i in range(0, n - window_size + 1, stride):
            if is_gap[i:i+window_size].any():
                continue
            data_arr_all.append(data_arr[i:i+window_size])
            cmdb_all.append(cmdb_id)
            ts_all.append(ts[i+window_size-1])
            y_all.append(int(labels[i:i+window_size].any()))
            w_types = sorted({t for t in ftypes[i:i+window_size] if isinstance(t, str)})
            ft_all.append(','.join(w_types) if w_types else None)
    X = np.stack(data_arr_all)
    return X, np.array(cmdb_all), np.array(ts_all), np.array(y_all, dtype=np.int64), np.array(ft_all, dtype=object)

raw_windows = {}
for name, fname in SPLIT_FILES.items():
    d = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)
    d['is_gap'] = d['is_gap'].astype(bool)
    X, cmdb, ts, y, ft = build_windows_full(d, FEATURE_COLS, WINDOW_SIZE)
    X_flat = X.reshape(len(X), -1)
    X_pca = np.clip(pca.transform(X_flat), -CLIP, CLIP).astype(np.float32)
    raw_windows[name] = {'X': X_pca, 'cmdb': cmdb, 'ts': ts, 'y': y, 'ft': ft}
    print(f'  {name:10s}: {len(y):>7,} windows  ({int(y.sum()):,} anomalies)')

with torch.no_grad():
    mse_val = base_model.anomaly_score(torch.from_numpy(raw_windows['cc1_val']['X'])).numpy()
print(f'\nSanity check: val_p99 recomputed = {np.percentile(mse_val, 99):.5f}  (should match {VAL_P99:.5f})')

  cc1_train : 152,456 windows  (0 anomalies)
  cc1_val   :  20,763 windows  (0 anomalies)
  cc1_test  :  43,375 windows  (256 anomalies)
  drift_cc2 :  76,167 windows  (1,080 anomalies)

Sanity check: val_p99 recomputed = 1.44733  (should match 1.44733)


## Step 2 — Recalibrate `PRIOR_STRENGTH` with the corrected `k`

Same procedure as before, but using `K_CALIBRATED` instead of the fixed 3.0
— this should now converge to ~1% FPR as prior_strength grows, by
construction, since the global endpoint now equals `val_p99` exactly.

In [3]:
def run_blended(mse_arr, cmdb_arr, prior_strength, k, buffer_size=BUFFER_SIZE,
                 global_mean=GLOBAL_MEAN, global_std=GLOBAL_STD):
    n = len(mse_arr)
    preds = np.zeros(n, dtype=np.int64)
    buffers = {}
    for i in range(n):
        cid = cmdb_arr[i]
        buf = buffers.setdefault(cid, deque(maxlen=buffer_size))
        n_local = len(buf)
        w = n_local / (n_local + prior_strength)
        if n_local == 0:
            lm, ls = global_mean, global_std
        else:
            arr = np.fromiter(buf, dtype=np.float64); lm, ls = arr.mean(), arr.std()
        t = (w * lm + (1 - w) * global_mean) + k * (w * ls + (1 - w) * global_std)
        is_anom = mse_arr[i] > t
        preds[i] = int(is_anom)
        if not is_anom:
            buf.append(mse_arr[i])
    return preds

PRIOR_CANDIDATES = [100, 500, 2000, 10000, 50000]
print(f'{"prior_strength":>14s} {"FPR (old k=3.0)":>16s} {"FPR (calibrated k)":>20s}')
prior_fpr_new = {}
for ps in PRIOR_CANDIDATES:
    fpr_old = run_blended(mse_val, raw_windows['cc1_val']['cmdb'], ps, K_OLD).mean()
    fpr_new = run_blended(mse_val, raw_windows['cc1_val']['cmdb'], ps, K_CALIBRATED).mean()
    prior_fpr_new[ps] = fpr_new
    print(f'{ps:14d} {fpr_old*100:15.2f}% {fpr_new*100:19.2f}%')

BEST_PRIOR = min(PRIOR_CANDIDATES, key=lambda ps: abs(prior_fpr_new[ps] - 0.01))
print(f'\nChosen PRIOR_STRENGTH (with calibrated k) = {BEST_PRIOR}')

prior_strength  FPR (old k=3.0)   FPR (calibrated k)
           100            2.28%                1.32%
           500            2.03%                1.10%
          2000            1.92%                1.02%
         10000            1.90%                1.00%
         50000            1.91%                0.99%

Chosen PRIOR_STRENGTH (with calibrated k) = 10000


## Step 3 — Re-evaluate: VAE alone vs. + Adaptive (fixed) vs. Full Model (fixed)

In [4]:
with torch.no_grad():
    mse_cache = {name: base_model.anomaly_score(torch.from_numpy(raw_windows[name]['X'])).numpy() for name in ALL_SETS}

vae_alone_f1 = {}
for name in ALL_SETS:
    pred = (mse_cache[name] > VAL_P99).astype(int)
    vae_alone_f1[name] = f1_score(raw_windows[name]['y'], pred, zero_division=0)

adaptive_fixed_results = {}
for name in ALL_SETS:
    preds = run_blended(mse_cache[name], raw_windows[name]['cmdb'], BEST_PRIOR, K_CALIBRATED)
    y = raw_windows[name]['y']
    p, r, f1 = precision_score(y, preds, zero_division=0), recall_score(y, preds, zero_division=0), f1_score(y, preds, zero_division=0)
    adaptive_fixed_results[name] = {'precision': p, 'recall': r, 'f1': f1}
    print(f'{name:12s} + Adaptive (FIXED k)   Precision={p:.3f}  Recall={r:.3f}  F1={f1:.3f}   (VAE alone F1={vae_alone_f1[name]:.3f})')

cc1_test     + Adaptive (FIXED k)   Precision=0.986  Recall=0.848  F1=0.912   (VAE alone F1=0.912)
drift_cc2    + Adaptive (FIXED k)   Precision=0.089  Recall=0.806  F1=0.159   (VAE alone F1=0.155)


In [5]:
REFIT_INTERVAL, FT_BUFFER_SIZE, FT_LR, FT_EPOCHS, KS_ALPHA = 5000, 2000, 1e-4, 5, 0.001

rng = np.random.default_rng(42)
ref_idx = rng.choice(len(raw_windows['cc1_train']['X']), size=5000, replace=False)
REF_SAMPLE = raw_windows['cc1_train']['X'][ref_idx]
with torch.no_grad():
    REFERENCE_MSE = base_model.anomaly_score(torch.from_numpy(REF_SAMPLE)).numpy()

def run_stream_fixed(model, X_stream, cmdb_stream, k, prior_strength, reference_mse):
    model = copy.deepcopy(model)
    opt = torch.optim.Adam(model.parameters(), lr=FT_LR)
    preds = np.zeros(len(X_stream), dtype=np.int64)
    threshold_buffers = {}
    ft_pool = deque(maxlen=FT_BUFFER_SIZE)
    n_finetunes, finetune_events = 0, []

    for i in range(len(X_stream)):
        x_i = X_stream[i]; cid = cmdb_stream[i]
        x_t = torch.from_numpy(x_i).unsqueeze(0)
        with torch.no_grad():
            mse = model.anomaly_score(x_t).item()

        buf = threshold_buffers.setdefault(cid, deque(maxlen=BUFFER_SIZE))
        n_local = len(buf)
        w = n_local / (n_local + prior_strength)
        if n_local == 0:
            lm, ls = GLOBAL_MEAN, GLOBAL_STD
        else:
            arr = np.fromiter(buf, dtype=np.float64); lm, ls = arr.mean(), arr.std()
        t = (w * lm + (1 - w) * GLOBAL_MEAN) + k * (w * ls + (1 - w) * GLOBAL_STD)

        is_anom = mse > t
        preds[i] = int(is_anom)
        if not is_anom:
            buf.append(mse); ft_pool.append(x_i)

        if (i + 1) % REFIT_INTERVAL == 0 and len(ft_pool) >= FT_BUFFER_SIZE // 2:
            with torch.no_grad():
                recent_mse = model.anomaly_score(torch.from_numpy(np.stack(list(ft_pool)))).numpy()
            _, p_value = stats.ks_2samp(reference_mse, recent_mse)
            if p_value < KS_ALPHA:
                Xb = torch.from_numpy(np.stack(list(ft_pool)))
                model.train()
                for _ in range(FT_EPOCHS):
                    opt.zero_grad()
                    recon, mu, logvar = model(Xb)
                    recon_loss = nn.functional.mse_loss(recon, Xb, reduction='mean')
                    kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
                    (recon_loss + meta60['beta_max'] * kl).backward()
                    opt.step()
                model.eval()
                n_finetunes += 1
                finetune_events.append(i + 1)
    return preds, n_finetunes, finetune_events

full_model_fixed_results = {}
for name in ALL_SETS:
    order = np.argsort(raw_windows[name]['ts'], kind='stable')
    X_stream = raw_windows[name]['X'][order]
    cmdb_stream = raw_windows[name]['cmdb'][order]
    y_stream = raw_windows[name]['y'][order]

    preds_full, n_ft, ft_events = run_stream_fixed(base_model, X_stream, cmdb_stream, K_CALIBRATED, BEST_PRIOR, REFERENCE_MSE)
    p, r, f1 = precision_score(y_stream, preds_full, zero_division=0), recall_score(y_stream, preds_full, zero_division=0), f1_score(y_stream, preds_full, zero_division=0)
    full_model_fixed_results[name] = {'precision': p, 'recall': r, 'f1': f1, 'n_finetunes': n_ft}
    print(f'{name:12s} Full Model (FIXED k)   Precision={p:.3f}  Recall={r:.3f}  F1={f1:.3f}  (fine-tunes: {n_ft})')

cc1_test     Full Model (FIXED k)   Precision=0.986  Recall=0.848  F1=0.912  (fine-tunes: 6)
drift_cc2    Full Model (FIXED k)   Precision=0.092  Recall=0.800  F1=0.165  (fine-tunes: 15)


## Step 4 — Full comparison: old (broken) k=3.0 vs. fixed calibrated k, at window=60

In [6]:
print(f'{"set":12s} {"config":24s} {"VAE alone":>10s} {"OLD k=3.0":>10s} {"FIXED k":>10s}')
old_results = {
    'cc1_test':  {'adaptive': 0.767, 'full_model': 0.768},
    'drift_cc2': {'adaptive': 0.110, 'full_model': 0.115},
}
for name in ALL_SETS:
    print(f'{name:12s} {"VAE alone":24s} {vae_alone_f1[name]:10.3f} {"--":>10s} {"--":>10s}')
    print(f'{name:12s} {"+ Adaptive":24s} {"--":>10s} {old_results[name]["adaptive"]:10.3f} {adaptive_fixed_results[name]["f1"]:10.3f}')
    print(f'{name:12s} {"Full Model":24s} {"--":>10s} {old_results[name]["full_model"]:10.3f} {full_model_fixed_results[name]["f1"]:10.3f}')
    print()

set          config                    VAE alone  OLD k=3.0    FIXED k
cc1_test     VAE alone                     0.912         --         --
cc1_test     + Adaptive                       --      0.767      0.912
cc1_test     Full Model                       --      0.768      0.912

drift_cc2    VAE alone                     0.155         --         --
drift_cc2    + Adaptive                       --      0.110      0.159
drift_cc2    Full Model                       --      0.115      0.165



## Step 5 — Save

In [7]:
save_results = {
    'window_size': WINDOW_SIZE,
    'k_old': K_OLD, 'k_calibrated': float(K_CALIBRATED),
    'best_prior_strength': BEST_PRIOR,
    'vae_alone_f1': vae_alone_f1,
    'adaptive_fixed': adaptive_fixed_results,
    'full_model_fixed': full_model_fixed_results,
    'old_broken_results': old_results,
}
out_path = os.path.join(OUT_DIR, 'window_size_60_threshold_fix_results.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {out_path}')

Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\experiments\window_size_60_threshold_fix_results.pkl


## How to read this

**Step 4 is the honest verdict**: does deriving `k` from the model's own
error distribution (rather than assuming a fixed 3.0) fix the adaptive
threshold's calibration problem at window=60? If `FIXED k`'s F1 now matches
or exceeds `VAE alone`'s F1 on both sets (rather than the OLD k's clear
regression on both), the fix worked and the full mechanism (adaptive
threshold + incremental learning) can be used with window=60 as originally
intended, not abandoned in favor of VAE-alone only.